# 00 — Pipeline Quickstart

The fastest way to verify the full data pipeline end-to-end.  
Run this after `python scripts/preprocess.py --config configs/preprocess_qm9.yaml`.

Each cell is a standalone check — run them in order or jump to whichever one is failing.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'IG-MPNN' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'Root: {ROOT}')

## Check 1 — Imports

In [ ]:
import torch
import torch_geometric
import rdkit
import numpy, scipy, pandas, sklearn

print(f'torch          {torch.__version__}')
print(f'torch_geometric {torch_geometric.__version__}')
print(f'rdkit          {rdkit.__version__}')
print(f'numpy          {numpy.__version__}')
print('All core imports OK ✓')

## Check 2 — Dataset loads

In [ ]:
from src.data import QM9Dataset

ds = QM9Dataset(root=ROOT / 'data', target_idx=0)
print(ds)
assert len(ds) > 100_000, 'QM9 should have ~130k molecules'
print('Dataset OK ✓')

## Check 3 — Single graph shapes

In [ ]:
d = ds[0]

checks = [
    ('x',              d.x.dim() == 2 and d.x.shape[1] == 11,     f'Expected (N,11), got {d.x.shape}'),
    ('edge_index',     d.edge_index.shape[0] == 2,                  f'Expected (2,E), got {d.edge_index.shape}'),
    ('pos',            d.pos.shape[1] == 3,                         f'Expected (N,3), got {d.pos.shape}'),
    ('y scalar',       d.y.numel() == 1,                            f'Expected scalar, got {d.y.shape}'),
    ('edge_attr_geo',  hasattr(d, 'edge_attr_geo'),                 'edge_attr_geo missing — GeometryTransform not applied?'),
    ('angle_attr',     hasattr(d, 'angle_attr'),                    'angle_attr missing'),
    ('torsion_attr',   hasattr(d, 'torsion_attr'),                  'torsion_attr missing'),
]

all_ok = True
for name, ok, msg in checks:
    status = '✓' if ok else '✗'
    print(f'  [{status}] {name:<18}  {msg if not ok else ""}')
    if not ok:
        all_ok = False

if all_ok:
    print('\nAll shape checks passed ✓')

## Check 4 — Split file exists and is consistent

In [ ]:
import json

split_file = ROOT / 'data' / 'splits' / 'qm9_split.json'
assert split_file.exists(), f'Split file not found: {split_file}\nRun scripts/preprocess.py first.'

with open(split_file) as f:
    payload = json.load(f)

train_idx = payload['train_indices']
test_idx  = payload['test_indices']

print(f"Train: {len(train_idx):,}  Test: {len(test_idx):,}")
print(f"Ratio: {len(train_idx)/(len(train_idx)+len(test_idx)):.0%} / {len(test_idx)/(len(train_idx)+len(test_idx)):.0%}")
print(f"Seed:  {payload['seed']}")

overlap = set(train_idx) & set(test_idx)
assert len(overlap) == 0, f'{len(overlap)} indices appear in both splits!'

all_covered = sorted(train_idx + test_idx) == list(range(len(ds)))
assert all_covered, 'Some molecules are missing from the split!'

print('Split file OK ✓')

## Check 5 — DataLoaders iterate

In [ ]:
from src.data import get_loaders

train_loader, test_loader = get_loaders(
    dataset=ds,
    train_idx=train_idx,
    test_idx=test_idx,
    batch_size=32,
)

tb = next(iter(train_loader))
vb = next(iter(test_loader))

print(f'Train batch: {tb.num_graphs} graphs, x={tb.x.shape}, y={tb.y.shape}')
print(f'Test  batch: {vb.num_graphs} graphs, x={vb.x.shape}, y={vb.y.shape}')
print(f'Train batches/epoch: {len(train_loader)}')
print(f'Test  batches/epoch: {len(test_loader)}')
print('DataLoaders OK ✓')

## Check 6 — Geometry values in expected ranges

In [ ]:
import math

# Quick check on 100 molecules
bond_ok, angle_ok, torsion_ok = True, True, True
for i in range(100):
    d = ds[i]
    if hasattr(d, 'edge_attr_geo'):
        if d.edge_attr_geo.min().item() < 0:
            bond_ok = False
        if d.edge_attr_geo.max().item() > 5.0:   # no QM9 bond > 5 Å
            bond_ok = False
    if hasattr(d, 'angle_attr'):
        if d.angle_attr.min().item() < 0 or d.angle_attr.max().item() > math.pi + 1e-4:
            angle_ok = False
    if hasattr(d, 'torsion_attr'):
        if d.torsion_attr.min().item() < -(math.pi + 1e-4) or d.torsion_attr.max().item() > math.pi + 1e-4:
            torsion_ok = False

print(f'  Bond lengths in [0, 5] Å     : {"✓" if bond_ok else "✗"}')
print(f'  Bond angles in [0, π] rad    : {"✓" if angle_ok else "✗"}')
print(f'  Torsions in [-π, π] rad      : {"✓" if torsion_ok else "✗"}')

if bond_ok and angle_ok and torsion_ok:
    print('Geometry range checks OK ✓')

## Summary

In [ ]:
print('=' * 50)
print('  Pipeline status')
print('=' * 50)
print(f'  Dataset     : {ds}')
print(f'  Train split : {len(train_idx):,} molecules')
print(f'  Test split  : {len(test_idx):,} molecules')
print(f'  Batch size  : 32')
print(f'  Node feat   : {ds.num_node_features}-dim')
print(f'  Bond feat   : {ds.num_edge_features}-dim')
print(f'  Geometry    : lengths + angles + torsions')
print('=' * 50)
print('  Ready to train.')